In [1]:
import pandas as pd
import re

# ================= IEEE =================
def parse_ieee(texte):
    articles = texte.split("\n\n")
    data = []

    for article in articles:
        if not article.strip():
            continue

        lignes = article.strip().split("\n")
        ligne_principale = lignes[0]

        # Auteurs
        auteurs = ligne_principale.split('",')[0]

        # Titre
        title_match = re.search(r'"(.*?)"', ligne_principale)
        titre = title_match.group(1) if title_match else ""

        # Journal
        journal_match = re.search(r'in (.*?), vol', ligne_principale)
        journal = journal_match.group(1) if journal_match else ""

        # Année
        annee_match = re.search(r', (\d{4}),', ligne_principale)
        annee = annee_match.group(1) if annee_match else ""

        # DOI
        doi_match = re.search(r'doi: ([^\.\n]+)', ligne_principale)
        doi = doi_match.group(1) if doi_match else ""

        # Mots-clés
        mots_cles = ""
        if len(lignes) > 1:
            kw_match = re.search(r'keywords: {(.*?)}', lignes[1])
            if kw_match:
                mots_cles = kw_match.group(1)

        data.append({
            "Title": titre,
            "Authors": auteurs,
            "Year": annee,
            "Journal": journal,
            "Abstract": "",
            "DOI": doi,
            "Keywords": mots_cles,
            "Source_DB": "IEEE"
        })

    return pd.DataFrame(data)




# ================= SCOPUS =================
def parse_scopus(texte):
    articles = re.split(r'\n(?=[A-Z][a-zA-Z\-]+.*,)', texte)
    data = []

    for article in articles:
        if "DOI" not in article:
            continue

        lignes = article.strip().split("\n")

        auteurs = lignes[0].strip()
        titre = lignes[2].strip() if len(lignes) > 2 else ""

        annee = ""
        journal = ""

        match = re.search(r'\((\d{4})\)\s*(.*)', article)
        if match:
            annee = match.group(1)
            journal = match.group(2).split(",")[0]

        doi_match = re.search(r'DOI:\s*(\S+)', article)
        doi = doi_match.group(1) if doi_match else ""

        abstract_match = re.search(
            r'ABSTRACT:\s*(.*?)(?:AUTHOR KEYWORDS:|INDEX KEYWORDS:)',
            article,
            re.S
        )
        abstract = abstract_match.group(1).strip() if abstract_match else ""

        keywords_match = re.search(r'AUTHOR KEYWORDS:\s*(.*)', article)
        keywords = keywords_match.group(1).strip() if keywords_match else ""

        data.append({
            "Title": titre,
            "Authors": auteurs,
            "Year": annee,
            "Journal": journal,
            "Abstract": abstract,
            "DOI": doi,
            "Keywords": keywords,
            "Source_DB": "Scopus"
        })

    return pd.DataFrame(data)

In [2]:
with open("downloads/scopus_export_Mar 30-2026_31c9e799-c6ca-4a69-bbd8-3bd2a75ca1fa.txt", "r", encoding="utf-8") as f:
    scopus_text = f.read()

with open("downloads/IEEE Xplore Citation Plain Text Download 2026.3.30.19.20.42.txt", "r", encoding="utf-8") as f:
    ieee_text = f.read()


In [4]:
df_scopus = parse_scopus(scopus_text)
df_ieee = parse_ieee(ieee_text)

df_all = pd.concat([df_scopus, df_ieee], ignore_index=True)

In [5]:
!pip install openpyxl

In [6]:
# ================= NORMALIZATION =================

def normalize_doi(doi):
    if not doi:
        return None
    
    doi = str(doi).lower().strip()

    if doi in ["", "nan", "none"]:
        return None

    return doi.replace("https://doi.org/", "")

def normalize_title(title):
    if not title:
        return ""
    
    title = title.lower()
    title = re.sub(r'[^a-z0-9 ]', ' ', title)
    title = re.sub(r'\s+', ' ', title).strip()
    
    return title

df_all["DOI_clean"] = df_all["DOI"].apply(normalize_doi)
df_all["Title_clean"] = df_all["Title"].apply(normalize_title)

# ================= DEBUG (IMPORTANT) =================



# ================= SPLIT =================

df_with_doi = df_all[df_all["DOI_clean"].notna() & (df_all["DOI_clean"] != "")]
df_no_doi = df_all[df_all["DOI_clean"].isna() | (df_all["DOI_clean"] == "")]



# ================= DEDUP =================

df_with_doi = df_with_doi.drop_duplicates(subset=["DOI_clean"], keep="first")

df_no_doi = df_no_doi.drop_duplicates(subset=["Title_clean"], keep="first")

# ================= MERGE =================

df_clean = pd.concat([df_with_doi, df_no_doi], ignore_index=True)

print("FINAL:", len(df_clean))

# ================= SAVE =================

df_clean.to_excel("downloads/RGB-T_Clean.xlsx", index=False)

FINAL: 85
